## IMPORT LIBRAIRIES

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

## IMPORT DATASETS

In [ ]:
df_primary = pd.read_csv("data/table_dataset/primary_data.csv", sep=";")
df_secondary = pd.read_csv("data/table_dataset/secondary_data.csv", sep=";")
df_mushnames = pd.read_csv("table_dataset/species_name.csv", sep =";")

### Cleaning data tabular 

#### Data for edible classification

In [ ]:
# Species name from primary, add to secondary
primary_name_df = df_primary[['family','name']]
df_primname_rep = primary_name_df.loc[primary_name_df.index.repeat(353)].reset_index(drop=True)

# Clean names
data_secondary_labelled = pd.concat([df_secondary, df_primname_rep], axis=1)
data_secondary_labelled['family'] = data_secondary_labelled['family'].str.replace(" Family", "", regex=False)
#
data_secondary_labelled['Common Name'] = data_secondary_labelled["family"] + " " + data_secondary_labelled["name"]
data_secondary_labelled

# Merge to scientific names
data_merge_scname = data_secondary_labelled.merge(df_mushnames, how='left', on='Common Name')
data_tabular_final = data_merge_scname.drop(columns=['family','name','Common Name'])
data_tabular_final.columns = data_tabular_final.columns.str.replace('-', '_').str.replace(' ', '_').str.lower()
data_tabular_final["class"] = (data_tabular_final["class"] == "p").astype(int)
data_tabular_final.info()

#### Data for species classification

In [ ]:
## Liste champignons (via noms scientifiques) dans les images
path_edible = "data/image_dataset/edible"
path_poisonous = "data/image_dataset/poisonous"

# Liste des sous-dossiers
list_edible = [f for f in os.listdir(path_edible)
                 if os.path.isdir(os.path.join(path_edible, f))]
list_poisonous = [f for f in os.listdir(path_poisonous)
                 if os.path.isdir(os.path.join(path_poisonous, f))]

# Création du DataFrame
df1 = pd.DataFrame(list_edible, columns=["scientific_name"])
df1['type'] = "edible"
df2 = pd.DataFrame(list_poisonous, columns=["scientific_name"])
df2['type'] = "poisonous"
df_concat = pd.concat([df1, df2], ignore_index=True)
df_concat['scientific_name'] = df_concat['scientific_name'].str.replace("_", " ", regex=False).str.replace("-", " ", regex=False)

# garder que le tabulaire dont l'espèce est présente dans les données d'image
data_tabular_image = data_tabular_final.merge(df_concat, how='inner', on='scientific_name')
data_tabular_image

### Train Test Split both datas

In [ ]:
# EDIBLE CLASSIFICATION
# prepare X and y

X = data_tabular_final.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y = data_tabular_final['class']

# TTS

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=3)

In [ ]:
# SPECIES CLASSIFICATION
# prepare X and y

X_tabimage = data_tabular_image.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y_tabimage = data_tabular_image['scientific_name']

# TTS

X_tabimage_train, X_tabimage_test, y_tabimage_train, y_tabimage_test = train_test_split(X_tabimage,
                                                                            y_tabimage,
                                                                            test_size=0.3,
                                                                            random_state=3)

### Preprocess pipeline

In [ ]:
# pipeline num and cat
num_transformer = make_pipeline(MinMaxScaler())
cat_transformer = make_pipeline(SimpleImputer(strategy='constant', fill_value='u'),
                                OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False))

# preprocess all
preproc_basic = make_column_transformer(
    (num_transformer, make_column_selector(dtype_include=np.number)),
    (cat_transformer, make_column_selector(dtype_exclude=np.number)),
    remainder='drop'
).set_output(transform="pandas")


# Apply preproc
X_train_preprocessed = preproc_basic.fit_transform(X_train)
X_test_preprocessed = preproc_basic.transform(X_test)
print(X_train_preprocessed.shape)
print(X_test_preprocessed.shape)